### 프롬프트 엔지니어링
1. 명확하고 구체적으로 써야 한다.
    - 틀린 예)
        - 적당한 길이로 답변해(X) -> 100자 내외로 답변해(O)
        - 예쁘게 답변해(X) -> 예시를 줘 가면서 답변하라고 시켜야! (퓨 샷 기법)


2. 명확하게 쓰기 어려울 때는 예시를 줘야 한다. (few shot)
    - 예) 경상도 사투리로 대답해달라 할때 예시 주기
        - 밥 먹었니 -> 밥 먹었나 , 많이 힘들다 -> 대다

3. 출력 형식 : 결과물을 json 형태로 받는 게 좋다!
    - 나중에 결과물 저장하기 좋게 -> 데이터베이스에 넣거나 처리하기 좋게 key : value의 형식으로 받는 게 좋다

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [2]:
# 요청하는 함수를 만들어서 재사용 - 시스템 프롬프트를 적용한 단일 턴 대화
def ask(user, system=None, model="gpt-5.6-luna"):
    messages = []
    if system:
        messages.append({"role" : "system", "content" : system})

    messages.append({"role" : "user", "content" : user})

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [4]:
question = "간단한 이력서를 작성해줘"

result = ask(question)
print(result)

아래 양식을 참고해 작성할 수 있습니다.

# 이력서

## 1. 기본 정보
- 이름: 홍길동
- 연락처: 010-1234-5678
- 이메일: example@email.com
- 주소: 서울특별시 ○○구

## 2. 지원 분야
- 지원 직무: 마케팅 / 사무직 / 개발자 등

## 3. 학력
- 20XX.03 ~ 20XX.02  ○○대학교 ○○학과 졸업
- 20XX.03 ~ 20XX.02  ○○고등학교 졸업

## 4. 경력
- 20XX.03 ~ 20XX.12  ○○회사 / ○○팀
  - 담당 업무: 고객 관리, 문서 작성, 프로젝트 지원
  - 주요 성과: 업무 효율 20% 향상

## 5. 보유 역량
- MS Office 활용 가능
- 문서 작성 및 자료 정리
- 원활한 의사소통 능력
- 책임감 있고 꼼꼼한 업무 처리

## 6. 자격증 및 교육
- 컴퓨터활용능력 2급
- 워드프로세서
- 관련 교육 수료: ○○교육 과정

## 7. 자기소개
책임감과 성실함을 바탕으로 맡은 업무를 끝까지 수행합니다. 새로운 업무를 빠르게 배우고, 원활한 소통을 통해 조직에 기여하는 인재가 되겠습니다.


In [5]:
# 파일 저장해서 확인해보기
with open("result.md", "w", encoding="utf-8-sig") as file:
    file.write(result) 

In [ ]:
system_prompt = """
너는 이력서 담당해주는 코칭 스탭이야.
채용 담당자가 볼 수 있도록 매력적으로 느껴질 수 있는 이력서를 잘 작성해.
"""

### 1. 지시가 구체적으로 작성되도록
- 모호한 지시와 구체적 지시의 결과물 비교

In [6]:
# 홍보 문구 점검
text = "우리 재생크림은 손상된 피부를 즉각적으로 재생시켜줍니다. 가격도 괜찮습니다."

result = ask("이 글 어때?" + text)
print(result)

의미는 전달되지만, 광고 문구로는 다소 과장되고 표현이 모호합니다.

- **“즉각적으로 재생시켜줍니다”**: 피부가 즉시 재생된다는 인상을 주어 과장 광고로 보일 수 있습니다.  
- **“가격도 괜찮습니다”**: 주관적이고 구체성이 부족합니다.

예를 들면 이렇게 다듬을 수 있습니다.

> **우리 재생 크림은 외부 자극으로 민감해진 피부를 편안하게 진정시키고, 촉촉한 보습을 제공합니다. 합리적인 가격으로 부담 없이 만나보세요.**

조금 더 광고 문구답게 쓰면:

> **손상으로 지친 피부에 깊은 보습과 편안한 진정 케어를 선사하는 재생 크림. 부담 없는 가격으로 매일 사용해 보세요.**

실제 임상 근거가 있다면 “피부 장벽 개선에 도움”처럼 근거에 맞는 표현을 사용하는 것이 좋습니다.


In [7]:
result = ask(f"""
다음 홍보 문구의 약점을 3가지 정도 나타내고, 각각 더 구체적으로 고친 예시를 3가지 정도 제시해줘.

[홍보문구]
{text}
""")

print(result)

다음 문구는 **효과의 과장, 근거 부족, 가격·대상 정보의 모호함**이 약점입니다. 화장품 홍보라면 ‘피부 재생’처럼 의약품으로 오인될 수 있는 표현은 특히 주의하는 것이 좋습니다.

## 1. 효과를 과장하고 표현이 모호함

**문제점**  
“손상된 피부를 즉각적으로 재생시켜준다”는 표현은 효과가 지나치게 강하고, ‘즉각적 재생’의 의미와 범위도 불분명합니다. 또한 실제 효능을 입증하지 못하면 과장 광고로 보일 수 있습니다.

**구체적으로 고친 예시**

1. “건조하고 민감해진 피부에 촉촉한 보습감을 선사해 피부 장벽 케어를 돕습니다.”
2. “세안 후 당기고 거칠어진 피부를 부드럽게 감싸 촉촉하게 관리해줍니다.”
3. “판테놀과 세라마이드를 함유해 외부 자극으로 민감해진 피부의 보습 장벽을 케어합니다.”

※ 실제 성분과 시험 결과가 있을 때만 해당 성분·효능을 사용해야 합니다.

---

## 2. 객관적인 근거나 구체적인 정보가 부족함

**문제점**  
어떤 성분이 들어 있는지, 어떤 피부 고민에 적합한지, 어느 정도의 사용감이나 효과를 기대할 수 있는지 알기 어렵습니다. 소비자가 제품의 차별점을 판단하기도 어렵습니다.

**구체적으로 고친 예시**

1. “판테놀과 세라마이드를 담아 건조한 피부에 보습을 더하고 피부 장벽을 촉촉하게 관리합니다.”
2. “끈적임은 줄이고 보습감은 높인 크림으로, 건조함과 거칠음이 고민인 피부에 적합합니다.”
3. “민감성 피부 사용 적합성 테스트를 완료한 저자극 보습 크림으로, 매일 부담 없이 사용할 수 있습니다.”

※ ‘저자극’, ‘테스트 완료’ 등의 문구는 실제 시험·검증 자료가 있을 때만 사용해야 합니다.

---

## 3. 가격 표현이 주관적이고 구매 정보가 부족함

**문제점**  
“가격도 괜찮습니다”는 사람마다 기준이 달라 설득력이 약합니다. 가격, 용량, 사용 기간, 할인 혜택 등 구매 판단에 필요한 정보가 빠져 있습니다.

**구체적으로 고친 예시**

1. “50mL 기준 19,800원으로

### 예시를 보여주기 - few shot
- 예시를 넣으면 더 좋은 경우

In [8]:
# 리뷰 메시지 감정 평가 시켜보기

review = "라로슈포제 화장품이 기대했던 것보다 훨씬 좋았어요. 피부 당김도 적어졌고, 피부에 광이 나는 느낌이 들어요"

result = ask(f"다음 리뷰의 감정을 판단해줘: {review}")
print(result)

긍정적인 감정입니다. 제품이 기대 이상으로 만족스러웠고, 피부 당김이 줄어들며 피부에 광이 나는 효과를 경험했다는 내용입니다.


In [9]:
few_ex = f"""
다음 리뷰의 감정을 긍정/부정/중립 중 하나로만 답하세요

# 예시
## [리뷰 1]
- 정말 최고에요! -> 긍정

## [리뷰 2]
- 최악이에요 -> 부정

## [리뷰 3]
- 고만고만해요 -> 중립

# 리뷰
{review}
"""

result = ask(few_ex)
print(result)

긍정


In [10]:
sentence = "오늘 점심 뭐 먹을지 고민이 많다."

few_ex = f"""
다음 표준어 문장을 자연스러운 전라도 사투리로 바꿔주세요.

변환 규칙:
- 문장의 뜻은 바꾸지 마세요.
- 전라도에서 자주 사용하는 말투와 어미를 사용하세요.
- 너무 과장되거나 알아보기 어려운 표현은 피하세요.
- 설명하지 말고, 변환된 문장만 출력하세요.

# 예시
## 문장1
- 정말 맛있어요! -> 아따, 참말로 맛있당께!

## 문장2
- 지금 어디에 가요? -> 지금 어디 가는디?

## 문장3
- 빨리 와 주세요. -> 얼른 와 주랑께.

## 문장4
- 오늘 날씨가 정말 좋네요. -> 오늘 날씨가 참말로 좋구마잉.

## 문장5
- 그렇게 하면 안 돼요. -> 그렇게 하믄 안 된당께.

# 변환할 문장
{sentence}
"""

result = ask(few_ex)
print(result)

오늘 점심 뭐 묵을랑가 고민이 많당께.


### 출력 형식 정해주기
- 프로그램에서 쓰려면 형식이 일정해야 합니다.

In [11]:
text = "점심 메뉴 추천"

result = ask(text)
print(result)

오늘 점심 메뉴 추천:

- **든든하게:** 제육볶음 + 계란찜  
- **깔끔하게:** 쌀국수 또는 냉모밀  
- **빠르게:** 김밥 + 라면  
- **건강하게:** 닭가슴살 샐러드나 포케  
- **스트레스 해소:** 마라탕 또는 돈가스  
- **비 오는 날:** 김치찌개나 칼국수  

하나만 고르라면 **제육볶음** 추천해요!


In [12]:
text = "점심 메뉴 추천"

result = ask(f"{text}을 - 로 시작하는 불릿 5개로만 답해줘")
print(result)

- 제육볶음
- 김치찌개
- 치킨 샐러드
- 냉모밀과 유부초밥
- 비빔밥


In [13]:
result = ask(f"{text}을 한 문장으로 말해줘")
print(result)

오늘 점심은 따뜻한 김치찌개와 계란말이로 든든하게 드세요.


### 프롬프트 템플릿 - 고정 규칙, 바뀌는 부분 -> 분리해놓기
- 같은 프롬프트를 매번 문장 바꿔서 넣는 대신에 -> 템플릿 형태로 만들어 놓는다
- 고정 규칙 : 역할, 형식, 금지사항 -> 템플릿
- 바뀌는 값: {자리} 에 넣어서 처리
- 값을 넣어서 최종 프롬프트 꼭 확인해보기

In [21]:
# 주제 - 쇼호스트
# 고정인 부분 - 소개하는 형식
# 바뀌는 부분 - 제품, 말투,

template = """
당신은  {tone} 말투의 제품 소개하는 쇼호스트입니다.
아래 상품을 5문장으로 소개하세요.
없는 기능이나 과장은 하지 마세요
마지막으로는 고객에게 추가 답변 유도할 수 있도록 질문으로 끝내 줘

## 상품
- {product}

## 제시 문장
"""

prompt = template.format(tone="전라도", product="무선 이어폰(노이즈 캔슬링, 24시간 배터리, 세련된 디자인)") # 이런 식으로도 할 수 있다

result = ask(prompt)
print(result)

요 무선 이어폰은 주변 소음을 줄여주는 노이즈 캔슬링 기능이 들어갔당께요.  
최대 24시간 배터리로 하루 동안 편하게 사용하기 좋겄지라.  
무선이라 선 엉킬 걱정 없이 간편하게 쓸 수 있어요.  
세련된 디자인이라 어디서든 깔끔하게 착용하기 좋구만요.  
노이즈 캔슬링 무선 이어폰 찾고 계셨던 거 맞으실까요?


In [22]:
prompt = template.format(tone="반말", product="재생크림(손상된 피부 회복, 미백 효과, 주름 개선)") # 이런 식으로도 할 수 있다

result = ask(prompt)
print(result)

손상된 피부를 편안하게 케어해주는 재생크림이야.  
피부 회복을 돕고, 칙칙한 피부 톤을 밝혀주는 미백 효과가 있어.  
주름 개선 기능으로 탄탄하고 매끄러운 피부 관리에도 도움을 줄 수 있어.  
피부가 지치고 거칠게 느껴질 때 꾸준히 사용해보면 좋아.  
손상 피부 회복과 미백, 주름 개선을 함께 관리하고 싶어?


### 단계별로 생각하게 하기 - Chain of Thought (CoT)
- 어려운 문제는 -> '차근차근 단계별로 생각해서 답해줘'라는 문구 필요하다
- gpt 5.6 luna는 기본적으로 생각하고 답하게 되어 있어서, 굳이 시키지 않아도 됨

### 파라미터 -> 신형 모델에서는 필요 X
- temperature
    - 창의성 정도

- top_p
    - 후보 단어

신형 모델에서는 다 프롬프트로 제어함

### 프롬프트 제어
- A/B -> 어떤 프롬프트가 더 좋은 결과물 만드는지 비교
- 자기일관성 -> 같은 질문을 여러 번 물어서 확인 -> 정답이 하나인 문제
    - 여러 번 물었는데 답이 다를 수 있는 경우 -> 다수결로 답 정하는 게 좋음
- 길이 제어: 짧게/길게 가 아니라, 몇자 이내가 좋음

#### 예시
- 하나도 없는 경우 : zero shot
- few shot
- 기준이 애매할 경우 -> 반드시 예시 넣어줘야
    - 말투, 품질 판단, 의견 등
    - 창작, 아이디어 -> 퓨 샷이지만 좀 더 다양한 예시를 들어야

#### 생각 필요한 문제
1. 정말 zero shot보다 few shot이 나을까
2. 우리 주제, 질문에 대해 규칙이 아주 세세한 게 나을까 적당한 게 나을까

In [ ]:
# 실습 - cdoex와 같이
# 1. 리뷰 분석하고 요구 사항 정리하는 프롬프트 만들어보기
template1 = """
당신은 사용자 리뷰를 분석해 제품 개선 요구사항을 정리하는 분석가입니다.

[분석 대상]
- 제품/서비스 주제: {주제}
- 분석 목표: 사용자 리뷰에서 반복되는 문제, 요구, 개선 기회를 찾아 실행 가능한 요구사항으로 정리한다.
- 리뷰 목록:
{리뷰_목록}

[작업 규칙]
1. 리뷰에 실제로 언급된 내용만 근거로 사용한다.
2. 추측이 필요한 내용은 사실처럼 쓰지 말고, “추가 확인 필요”로 표시한다.
3. 비슷한 의견은 묶고, 각 항목에 언급 빈도 또는 대표 리뷰 수를 표시한다.
4. 긍정 의견도 유지해야 할 요구사항으로 정리한다.
5. 요구사항은 개발자·기획자가 이해할 수 있게 구체적으로 작성하되, 해결 방법을 단정하지 않는다.
6. 중요도는 사용자 영향도와 반복 정도를 기준으로 높음·중간·낮음으로 분류한다.

[출력 형식]

## 1. 리뷰 핵심 요약
- 긍정 의견:
- 불편/문제:
- 사용자 요청:
- 추가 확인이 필요한 점:

## 2. 주요 이슈 분석
| 번호 | 이슈 | 근거가 된 리뷰 요약 | 빈도 | 영향도 | 중요도 |
|---|---|---|---|---|---|

## 3. 요구사항 정리
| ID | 요구사항 | 해결하려는 사용자 문제 | 우선순위 | 근거 |
|---|---|---|---|---|

## 4. 다음 행동 제안
- 가장 먼저 검토할 요구사항 3개
- 추가로 수집하거나 확인할 정보
- 리뷰만으로 결정하면 위험한 항목

## 5. 고객센터 답변
"""

reviews = """
1. 문제를 단계별로 풀 수 있어서 초보자도 따라가기 좋았습니다.
  2. 설명은 쉬운데, 정답을 너무 빨리 보여줘서 직접 생각할 시간이 부족했어요.
  3. 연습문제 난이도가 갑자기 높아지는 느낌입니다. 이전 문제와 연결되는 힌트가 있으면 좋겠어요.
  4. 코드 실행 결과를 바로 확인할 수 있어서 편리했습니다.
  5. 모바일에서는 코드 블록이 잘려서 읽기 어려웠습니다.
  6. 틀린 이유를 알려주는 피드백이 더 구체적이면 좋겠습니다. 지금은 “오답입니다”만 나와요.
  7. Python 기초 문법을 처음 배우는 사람에게 추천합니다. 설명이 부담스럽지 않았어요.
  8. 문제 수가 조금 더 많았으면 좋겠고, 반복 연습 기능도 필요해 보여요.
  9. 로그인한 뒤 학습 기록이 가끔 사라지는 것 같습니다.
  10. 디자인은 깔끔하지만, 다음에 무엇을 공부해야 할지 추천해 주면 더 좋겠습니다.
"""

In [29]:
prompt = template1.format(주제="사이트", 리뷰_목록=reviews) # 이런 식으로도 할 수 있다

result = ask(prompt)
print(result)

## 1. 리뷰 핵심 요약

- **긍정 의견:**
  - 문제를 단계별로 풀 수 있어 초보자도 따라가기 쉽다는 평가가 있습니다. (리뷰 1)
  - 설명이 쉽고 부담스럽지 않아 Python 기초 학습자에게 적합하다는 의견이 있습니다. (리뷰 2, 7)
  - 코드 실행 결과를 바로 확인할 수 있어 편리하다는 평가가 있습니다. (리뷰 4)
  - 디자인이 깔끔하다는 평가가 있습니다. (리뷰 10)

- **불편/문제:**
  - 정답이 너무 빨리 노출되어 스스로 생각할 시간이 부족합니다. (리뷰 2)
  - 연습문제 난이도가 갑자기 높아지고, 문제 간 연결 힌트가 부족합니다. (리뷰 3)
  - 모바일에서 코드 블록이 잘려 읽기 어렵습니다. (리뷰 5)
  - 오답 피드백이 구체적이지 않고 “오답입니다” 수준에 그칩니다. (리뷰 6)
  - 로그인 후 학습 기록이 사라지는 현상이 보고되었습니다. (리뷰 9)

- **사용자 요청:**
  - 문제 수 확대 및 반복 연습 기능이 필요합니다. (리뷰 8)
  - 다음에 학습할 내용을 추천해 주는 기능이 필요합니다. (리뷰 10)
  - 정답 노출 시점 조정, 단계별 힌트, 구체적인 오답 설명이 필요합니다. (리뷰 2, 3, 6)

- **추가 확인이 필요한 점:**
  - 학습 기록이 사라지는 빈도, 발생 조건, 영향받는 사용자 범위는 추가 확인이 필요합니다.
  - 모바일 코드 블록 문제가 발생하는 기기·운영체제·브라우저 범위는 추가 확인이 필요합니다.
  - 사용자가 원하는 문제 수, 반복 연습 방식, 학습 추천 기준은 추가 확인이 필요합니다.
  - “정답이 너무 빨리 보인다”는 문제가 모든 문제에 해당하는지, 특정 문제 유형에 해당하는지는 추가 확인이 필요합니다.

---

## 2. 주요 이슈 분석

| 번호 | 이슈 | 근거가 된 리뷰 요약 | 빈도 | 영향도 | 중요도 |
|---|---|---|---:|---|---|
| 1 | 단계별 학습과 쉬운 설명은 긍정적이나, 정답 노출 시점 조정이 필요함 |

In [30]:
# 2. 화장품 홍보 문구 만들어보기

product_info = """
  - 제품명: 수분 진정 크림
  - 제품 유형: 페이셜 크림
  - 핵심 성분/특징: 병풀 유래 성분, 산뜻한 사용감
  - 주요 고객: 건조함과 자극이 신경 쓰이는 20~30대
  - 원하는 분위기: 차분하고 믿음직한 느낌
  """

moderate_prompt = f"""
당신은 화장품 브랜드의 카피라이터입니다.
아래 제품 정보를 바탕으로 온라인 홍보 문구를 작성하세요.

[제품 정보]
{product_info}

[작성 요청]
1. 메인 홍보 문구 1개를 작성하세요. (25자 이내)
2. 제품 장점을 설명하는 문구 2개를 작성하세요. (각 50자 이내)
3. 구매를 유도하는 문구 1개를 작성하세요. (30자 이내)

[주의]
- 제품 정보에 없는 효능은 만들지 마세요.
- 치료, 완치, 의학적 효과를 보장하는 표현은 사용하지 마세요.
- 쉽고 자연스러운 한국어로 작성하세요.
"""

detailed_prompt = f"""
당신은 화장품 브랜드의 카피라이터입니다.
아래 제품 정보를 바탕으로 온라인 홍보 문구를 작성하세요.

[제품 정보]
{product_info}

[출력 형식]
아래 제목을 그대로 사용하고, 제목 외의 설명은 쓰지 마세요.

메인 문구:
보조 문구 1:
보조 문구 2:
구매 유도 문구:

[세부 규칙]
1. 메인 문구는 공백 포함 25자 이내로 작성합니다.
2. 보조 문구는 각각 공백 포함 50자 이내로 작성합니다.
3. 구매 유도 문구는 공백 포함 30자 이내로 작성합니다.
4. 각 문구에는 제품명 또는 제품 유형을 최소 1회 포함합니다.
5. 핵심 성분/특징 중 최소 1개를 반드시 반영합니다.
6. 타깃 고객의 고민 또는 사용 상황을 1개 반영합니다.
7. “최고”, “완벽”, “즉시”, “무조건”, “보장” 표현은 사용하지 않습니다.
8. 치료, 개선 보장, 의학적 효능을 암시하는 표현은 사용하지 않습니다.
9. 제품 정보에 없는 성분, 수치, 인증, 효능은 추가하지 않습니다.
10. 이모지, 해시태그, 느낌표는 사용하지 않습니다.
11. 문장 끝은 마침표 없이 작성합니다.
12. 서로 같은 표현을 반복하지 않습니다.
"""

result_moderate = ask(moderate_prompt)
result_detailed = ask(detailed_prompt)

print("=== 적당한 규칙 결과 ===")
print(result_moderate)

print("\n=== 세세한 규칙 결과 ===")
print(result_detailed)

=== 적당한 규칙 결과 ===
1. **메인 홍보 문구**  
건조한 날을 위한 산뜻한 수분 진정

2. **제품 장점 문구**  
- 병풀 유래 성분을 담은 수분 진정 크림입니다.  
- 산뜻한 사용감으로 매일 편안하게 사용할 수 있습니다.

3. **구매 유도 문구**  
오늘부터 산뜻한 수분 진정 케어를 시작하세요.

=== 세세한 규칙 결과 ===
메인 문구: 수분 진정 크림, 병풀로 산뜻하게
보조 문구 1: 건조함과 자극이 신경 쓰일 때 수분 진정 크림
보조 문구 2: 병풀 유래 성분을 담은 가벼운 페이셜 크림
구매 유도 문구: 건조한 날, 페이셜 크림으로 촉촉한 루틴
